Authenticate, create BigQuery client, and load the 1-week aggregated dataset (borg_traces_1w_jobmetrics) into pandas.

Note: we use the BigQuery Storage API for faster dataframe loading.

Output: df contains one row per instance execution.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np

client = bigquery.Client(project="traceadvisor-295a")

query = """
SELECT
  recurring_job_id,
  collection_id,
  instance_index,
  start_time,
  end_time,
  req_cpu,
  req_mem,
  peak_cpu,
  peak_mem
FROM `traceadvisor-295a.traceadvisor_eda.borg_traces_1w_jobmetrics`
WHERE start_time IS NOT NULL
  AND req_cpu IS NOT NULL AND req_mem IS NOT NULL
  AND peak_cpu IS NOT NULL AND peak_mem IS NOT NULL
"""
df = client.query(query).to_dataframe(create_bqstorage_client=True)
print(df.shape)

Compute split_t as a 70/30 time split from min/max start_time.

This ensures time-aware evaluation (train history precedes test).

In [ ]:
min_start = int(df["start_time"].min())
max_start = int(df["start_time"].max())
split_t = int(min_start + 0.7 * (max_start - min_start))
min_start, max_start, split_t

Cast large float columns to float32 to reduce Colab memory and speed up groupby quantile computation.

Safe because we are doing approximate decision metrics, not high-precision financial calculations.

In [ ]:
for c in ["req_cpu","req_mem","peak_cpu","peak_mem"]:
    df[c] = df[c].astype("float32")

Create execution_id and perform time-based split into train and test.

Ensures no leakage: test recommendations use only train history

In [ ]:
df["execution_id"] = df["collection_id"].astype(str) + "_" + df["instance_index"].astype(str)

train = df[df["start_time"] < split_t].copy()
test  = df[df["start_time"] >= split_t].copy()
print("Train:", train.shape, "Test:", test.shape)

def eval_method(df_eval, rec_cpu, rec_mem):
    rec_cpu = pd.Series(rec_cpu, index=df_eval.index)
    rec_mem = pd.Series(rec_mem, index=df_eval.index)

    viol_cpu = (df_eval["peak_cpu"] > rec_cpu).astype(int)
    viol_mem = (df_eval["peak_mem"] > rec_mem).astype(int)
    viol_any = ((viol_cpu == 1) | (viol_mem == 1)).astype(int)

    waste_cpu = np.maximum(0, rec_cpu - df_eval["peak_cpu"])
    waste_mem = np.maximum(0, rec_mem - df_eval["peak_mem"])

    return {
        "n_exec": int(len(df_eval)),
        "vr_cpu": float(viol_cpu.mean()),
        "vr_mem": float(viol_mem.mean()),
        "vr_any": float(viol_any.mean()),
        "waste_cpu": float(waste_cpu.sum()),
        "waste_mem": float(waste_mem.sum()),
    }

Evaluate Baseline 0 (recommendation equals original request) on the test slice.

Baseline 0 gives the “status quo” reliability and waste.

In [ ]:
b0 = eval_method(test, test["req_cpu"], test["req_mem"])
b0

Compute per-job P95 and P99 peak usage from train history.

Output includes n_hist used later for confidence tiers.

In [ ]:
min_history = 3  # matches "needs enough evidence" idea

q = train.groupby("recurring_job_id").agg(
    p95_cpu=("peak_cpu", lambda x: np.quantile(x, 0.95)),
    p99_cpu=("peak_cpu", lambda x: np.quantile(x, 0.99)),
    p95_mem=("peak_mem", lambda x: np.quantile(x, 0.95)),
    p99_mem=("peak_mem", lambda x: np.quantile(x, 0.99)),
    n_hist=("peak_cpu", "size")
).reset_index()

# Gate: if insufficient history, set quantiles to NaN so fallback-to-request triggers
mask = q["n_hist"] < min_history
q.loc[mask, ["p95_cpu", "p99_cpu", "p95_mem", "p99_mem"]] = np.nan

Apply P95 recommendations to test executions.

Fallback rule: if a job has no quantile history, use original request.

Evaluate violations + waste.

In [ ]:
test_q = test.merge(q, on="recurring_job_id", how="left")

# Recommended columns (like Meghana's "source" tracking)
test_q["rec_cpu_p95"] = test_q["p95_cpu"].fillna(test_q["req_cpu"])
test_q["rec_mem_p95"] = test_q["p95_mem"].fillna(test_q["req_mem"])
test_q["base_p95_source"] = np.where(test_q["p95_cpu"].isna(), "fallback_req", "history_p95")

test_q["rec_cpu_p99"] = test_q["p99_cpu"].fillna(test_q["req_cpu"])
test_q["rec_mem_p99"] = test_q["p99_mem"].fillna(test_q["req_mem"])
test_q["base_p99_source"] = np.where(test_q["p99_cpu"].isna(), "fallback_req", "history_p99")

Apply P99 recommendations and evaluate.

P99 should be safer (lower violations) than P95 but may increase waste.

In [ ]:
rec_cpu_p99 = test_q["p99_cpu"].fillna(test_q["req_cpu"])
rec_mem_p99 = test_q["p99_mem"].fillna(test_q["req_mem"])

b1_p99 = eval_method(test_q, rec_cpu_p99, rec_mem_p99)
b1_p99

Create a compact experiment summary for week-scale data.

Slack reduction computed relative to Baseline 0 waste.

This is the primary "result table" for workbook.

In [ ]:
b1_p95 = eval_method(test_q, test_q["rec_cpu_p95"], test_q["rec_mem_p95"])

summary = pd.DataFrame([
    {"method": "baseline0_as_is", **b0},
    {"method": "baseline1_p95", **b1_p95},
    {"method": "baseline1_p99", **b1_p99},
])

base_cpu = summary.loc[summary.method=="baseline0_as_is","waste_cpu"].iloc[0]
base_mem = summary.loc[summary.method=="baseline0_as_is","waste_mem"].iloc[0]

summary["slack_reduction_cpu_pct"] = 100*(base_cpu - summary["waste_cpu"])/(base_cpu + 1e-9)
summary["slack_reduction_mem_pct"] = 100*(base_mem - summary["waste_mem"])/(base_mem + 1e-9)

summary

Assign confidence tiers from n_hist and compute tiered results.

Reason: shows system behaves differently depending on evidence strength; supports "adoptability" argument.

In [ ]:
def tier(n):
    if pd.isna(n): return "low"
    n = int(n)
    if n >= 10: return "high"
    if n >= 3: return "medium"
    return "low"

test_q["tier"] = test_q["n_hist"].apply(tier)

tier_rows = []
for tier_name, g in test_q.groupby("tier"):
    rec_cpu = g["p95_cpu"].fillna(g["req_cpu"])
    rec_mem = g["p95_mem"].fillna(g["req_mem"])
    out = eval_method(g, rec_cpu, rec_mem)
    tier_rows.append({"method":"baseline1_p95", "tier":tier_name, **out})

tier_df = pd.DataFrame(tier_rows).sort_values("tier")
tier_df

Save summary CSV and tier CSV to Drive for easy sharing and later committing to repo under reports/eval/.

Note: raw data stays in BigQuery; only small outputs are stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

out_dir = "/content/drive/MyDrive/traceadvisor/results"
import os
os.makedirs(out_dir, exist_ok=True)

summary.to_csv(f"{out_dir}/summary_week.csv", index=False)
tier_df.to_csv(f"{out_dir}/tier_baseline1_p95_week.csv", index=False)

print("Saved results to:", out_dir)

In [ ]:
# Tier breakdown for P99
def tier(n):
    if pd.isna(n): return "low"
    n = int(n)
    if n >= 10: return "high"
    if n >= 3: return "medium"
    return "low"

test_q["tier"] = test_q["n_hist"].apply(tier)

tier_rows = []
for tier_name, g in test_q.groupby("tier"):
    rec_cpu = g["p99_cpu"].fillna(g["req_cpu"])
    rec_mem = g["p99_mem"].fillna(g["req_mem"])
    out = eval_method(g, rec_cpu, rec_mem)
    tier_rows.append({"method":"baseline1_p99", "tier":tier_name, **out})

tier_df_p99 = pd.DataFrame(tier_rows).sort_values("tier")
tier_df_p99.to_csv("/content/drive/MyDrive/traceadvisor/results/tier_baseline1_p99_week.csv", index=False)
tier_df_p99